# Week 8: An introduction to pandas

So far we have worked with tables by writing our own loops: a loop to read every row, a loop to count values, a loop to add things up. That's a great way to understand what is happening, but it takes a lot of code, and we keep writing the same patterns over and over.

**pandas** is a Python library made for working with tables. It does the looping for us, so a question that took ten lines of code can often be answered in one. In this notebook we'll use pandas to revisit a question from the Week 7 notes: **where were the Giller nominees born, and where do they live now?**

Then, starting in section 8, we'll use pandas to ask a new question: **how old are authors when they are first nominated?**

### Our data

We'll use the file `giller_authors.csv`, which we saved at the end of the Week 7 notes. It is the table we cleaned by hand last week:
- It has **one row per author**, not one row per book. 
- The regions have already been cleaned with our corrections dictionary.
- Unknown values are written as `Unknown`.

Because the cleaning is already done, we can focus on what pandas can do.

## 1. Loading the data

`import pandas as pd` loads the library. This is a convention that most people use to abbreviate pandas, so you will see `pd.` at the start of many pandas commands.

`pd.read_csv()` reads a whole CSV file in one step. Compare this with the `with open(...)` and `csv.DictReader` code we have been writing!

In [ ]:
import pandas as pd

authors = pd.read_csv("giller_authors.csv")

print(len(authors), "authors")
type(authors)

The variable `authors` has the pandas type `DataFrame`.  You can think of a DataFrame as a table with rows and columns. The big advantage of a specialized type like this is that there are a wide range of methods that operate on the table.  You will also see below that the DataFrame is displayed in a nicer way.

`authors.head()` shows us its first five rows.

In [ ]:
authors.head()

This looks a lot like a spreadsheet:
- The **column names** come from the header row of the CSV file.
- The numbers on the left (0, 1, 2, ...) are the **index**: pandas numbers the rows for us, starting from 0, just like a list.

## 2. Looking at one column

To get one column, we put its name in square brackets. This is just like getting a value from a dictionary, as in `book["Born"]`, except that we get the **whole column**: the value for every author at once.

Notice that pandas only shows the first and last few values of a long column, so the output stays readable.


A single column is called a **Series**.

In [ ]:
authors["Born region"]

## 3. Counting

Remember the counting loop we've written so many times?

```python
counts = {}
for author in birthplaces:
    region = ...
    if region in counts:
        counts[region] = counts[region] + 1
    else:
        counts[region] = 1
```

In pandas, that whole loop is one method: `.value_counts()`. It counts how many times each value appears in a column, and sorts the results from most to least common.

In [ ]:
authors["Born in Canada"].value_counts()

These are the same numbers we found in the Week 7 notes: 160 nominees born in Canada, 55 born elsewhere, and 3 unknown.

Now let's count the regions where the nominees were born:

In [ ]:
born_counts = authors["Born region"].value_counts()

born_counts

The result of `.value_counts()` is itself a Series, so we can save it in a variable, `born_counts`, and keep working with it. For example, `.head(10)` gives us just the first 10 values: the 10 most common regions.

In [ ]:
born_counts.head(10)

## 4. Visualizing the data

It is often easier to see differences between numbers with a chart rather than a list of numbers. Pandas can draw a chart from a Series with `.plot`. Behind the scenes it uses another library called **matplotlib**, which we import as `plt`.

`.plot.barh()` draws a **horizontal bar chart**. Horizontal bars work well when the labels are words, because the labels can be written sideways and stay easy to read.

In [ ]:
import matplotlib.pyplot as plt

born_counts.head(10).plot.barh()
plt.show()

This is a good start but there are some things we would like to fix:
- The biggest bar is at the **bottom**. `barh` draws the first value at the bottom, so we flip it.
- There is **no title**, and the numbers along the bottom have **no label**.

`.plot.barh()` gives back an object that we store in a variable called `ax` (short for "axes", matplotlib's word for one chart). We use `ax` to make these changes. 

In [ ]:
ax = born_counts.head(10).plot.barh(color="#2a78d6")

ax.invert_yaxis()   # put the biggest bar at the top
ax.set_title("Giller nominees by region of birth (top 10 regions)")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")   # the region names speak for themselves
plt.show()

## 5. The same question for residences

Now we can ask where the nominees live, by changing just the column name. (We'll use orange for residences, so a reader can tell the two charts apart.)

In [ ]:
lives_counts = authors["Lives region"].value_counts()

ax = lives_counts.head(10).plot.barh(color="#eb6834")

ax.invert_yaxis()
ax.set_title("Giller nominees by region of current residence (top 10)")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

Two separate charts make us flip back and forth. We can put both counts side by side in one chart. `pd.DataFrame` builds a table from a dictionary, where each key becomes a column name and each value is a Series. The two Series are lined up by their region names. A region that appears in only one Series gets a missing value (`NaN`), so we use `.fillna(0)` to turn those into 0.

To choose which regions to show, we add a `Total` column, which adds the two columns for every row. Then `.sort_values("Total", ascending=False)` puts the rows in order from biggest to smallest, and `.head(10)` keeps the first 10.

In [ ]:
both = pd.DataFrame({"Born": born_counts, "Lives": lives_counts}).fillna(0)
both["Total"] = both["Born"] + both["Lives"]

# Keep the 10 regions with the most births plus residences combined
top = both.sort_values("Total", ascending=False).head(10)

ax = top[["Born", "Lives"]].plot.barh(color=["#2a78d6", "#eb6834"])

ax.invert_yaxis()
ax.set_title("Giller nominees: region of birth vs. current residence")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

A chart of the top 10 regions leaves some authors out. How many?

`top.index` holds the labels of the rows of `top`: the names of our 10 regions. `.isin()` checks whether each value in a column is in that group of names, and gives back a column of `True` and `False`. The `~` in front flips every `True` to `False` and every `False` to `True`, so `~authors["Born region"].isin(top.index)` means "not one of the 10 regions".

In [ ]:
born_other = authors[~authors["Born region"].isin(top.index)]
lives_other = authors[~authors["Lives region"].isin(top.index)]

print(len(born_other), "authors were born outside the 10 regions in the chart")
print(len(lives_other), "authors live outside the 10 regions in the chart")

**Discussion:** Compare the two charts. Which regions have more nominees living there than were born there? Which have fewer? What might explain the difference?

## 6. Choosing rows: filtering

So far we have always used every row. Often we only want some of them: for example, only the nominees who were born outside Canada. In pandas, choosing rows is called **filtering**, and it takes two steps.

**Step 1:** Write a condition about a column. Just as `==` compares two values, here it compares *every* value in the column, and gives back a column of `True` and `False`.

You can think of this condition (or boolean expression) as a selector. When we use it like an index we get back only the rows that have True in our selector.

In [ ]:
born_elsewhere = authors["Born in Canada"] == "Elsewhere"

born_elsewhere

**Step 2:** Now `born_elsewhere` is our boolean selector, and when we apply it to the `authors` dataframe, we get a new DataFrame with only the rows in which authors were born somewhere other than Canada.

In [ ]:
non_canadians = authors[born_elsewhere]

print(len(non_canadians), "authors born outside Canada")
non_canadians.head()

`non_canadians` is a new DataFrame with the same columns as `authors`, but only 55 rows. Everything we've learned works on it too. Where were these nominees born?

In [ ]:
ax = non_canadians["Born region"].value_counts().head(20).plot.barh(color="#2a78d6")

ax.invert_yaxis()
ax.set_title("Nominees born outside Canada")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

Notice how we chained the steps together on one line: take the `"Born region"` column, count its values, keep the first 20, and plot them. You can always split a line like this into several steps with variables, as we did in section 3, if that's easier to read.

**Discussion:** The chart only shows the 20 most common places. How many different places are in the full list? (Try `len(non_canadians["Born region"].value_counts())`.) What does the chart hide about the variety of places Giller nominees come from?

Look closely at the bars after Hong Kong: they are all tied at 1, and the chart only has room for some of them, so there are more that we aren't seeing. Print `non_canadians["Born region"].value_counts()` to see them all. pandas had to break the tie somehow, so which places with one nominee made it into the chart is just luck. Would it be more fair to exclude every region that had only one nominee?

In [ ]:
# Let's split up these operations and plot all regions (countries) that
# have at least 2 nominees.

# First get the counts for each region from our non_canadians dataframe
non_can_counts = non_canadians["Born region"].value_counts()

# Then create a boolean selector for counts that are at least two
# This will give us a column of Trues and Falses that we can apply to
# select only the rows we want.
non_can_at_least_two = non_can_counts >= 2

# We can finally get the Series that we want to plot.
plotdata = non_can_counts[non_can_at_least_two]

ax = plotdata.plot.barh(color="#2a78d6")

ax.invert_yaxis()
ax.set_title("Nominees born outside Canada")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

## 7. From counts to percentages

So far we have counted authors. But a number like 160 is hard to judge by itself: is that most of the nominees, or only some of them? A **percentage** answers that, and it also lets us compare groups of different sizes.

`.value_counts(normalize=True)` gives each value's *fraction* of the rows, instead of its count:

In [ ]:
authors["Born in Canada"].value_counts(normalize=True)

These fractions add up to 1. To turn them into percentages, we multiply by 100. Notice that we don't need a loop: pandas does the calculation for every value in the Series. Then `.round(1)` rounds every value to one decimal place.

In [ ]:
born_percent = authors["Born in Canada"].value_counts(normalize=True) * 100
born_percent = born_percent.round(1)

born_percent

### Looking up one value

The result is a Series, and each value has a label on the left: `Canada`, `Elsewhere`, and `Unknown`. To get one value, we put its label in square brackets, just as we look up a key in a dictionary. This works on the result of a plain `.value_counts()` too.

Let's use a lookup to compare where the nominees were born with where they live now.

In [ ]:
lives_percent = authors["Lives in Canada"].value_counts(normalize=True) * 100
lives_percent = lives_percent.round(1)

print("Born in Canada:", born_percent["Canada"], "%")
print("Living in Canada:", lives_percent["Canada"], "%")

**Discussion:** More of the nominees live in Canada than were born here. What might explain that? The authors whose birthplace is `Unknown` are part of the total here, so they lower every other percentage a little. Would you leave them in or take them out?

## 8. How old are authors when they are first nominated?

Is the Giller a prize for established writers late in their careers, or does it find new voices? One way to approach this question is to ask **how old authors were when they were first nominated**.

Our spreadsheet has what we need: the `Year` of each nomination and each author's date of birth in the `DOB` column. This time we need one row per **book**, because each book has its own nomination year. So we load `giller_nominees_clean.csv` rather than `giller_authors.csv`, and keep only the columns we need. Two pairs of square brackets pick several columns at once.

In [ ]:
books = pd.read_csv("giller_nominees_clean.csv")
books = books[["Status", "Year", "Title", "Author", "DOB"]]

print(len(books), "nominated books")
books.head()

### Looking at the dates of birth

The dates in the `DOB` column are not all written the same way:
- a full date, like `09/06/1959` (day/month/year), sometimes without the zeros, like `4/4/1943`;
- just a month and year, like `10/1969`;
- just a year, like `1963`;
- a `-` when the date of birth is unknown.

Before we use a column, it's always worth checking for surprises. Here are two authors whose dates are written in different ways. `.isin()` keeps the rows whose author is in our list:

In [ ]:
books[books["Author"].isin(["Paulette Jiles", "Emma Donoghue"])]

Paulette Jiles's date of birth is written two different ways, and we only know the month and year for Emma Donoghue. That's not a problem for our question: every known date ends with the four-digit **year**, and the year is all we need.

### Keeping just the year

We've used slices to take part of a list or a string. A negative number counts from the end, so `[-4:]` means "from the fourth-last character to the end":

In [ ]:
dob = "09/06/1959"
print(dob[-4:])

To do that to every value in a column, we write `.str` first, just as we do with `.str.split()`. We save the result as a new column, `Birth year`.

In [ ]:
books["Birth year"] = books["DOB"].str[-4:]

books[books["Author"].isin(["Paulette Jiles", "Emma Donoghue"])]

**A warning about spreadsheets:** earlier versions of this file had `04-Apr-43` and `Oct-69` in these rows. Excel decided that these cells looked like dates and changed how they were written, throwing away the first two digits of the year! If you open a CSV file in Excel, check for this before you save it.

## 9. Calculating the ages

Some birth years are unknown, so we need to do three things:

1. **Leave out the unknown birth years.** We filter with `!=`, which means "not equal to". `.copy()` gives us a separate table of our own, which we'll add a column to.
2. **Turn the birth years into numbers.** They are text, like `"1959"`, because we cut them out of a string. `.astype(int)` converts a whole column, just as `int()` converts one value.
3. **Subtract.** Subtracting one column from another subtracts the values row by row, so one line calculates the age for every book.

In [ ]:
known_books = books[books["Birth year"] != "-"].copy()

known_books["Birth year"] = known_books["Birth year"].astype(int)

known_books["Age"] = known_books["Year"] - known_books["Birth year"]

print(len(books) - len(known_books), "books left out because the date of birth is unknown")
known_books[["Year", "Title", "Author", "Age"]].head()

These ages are approximate: we know the *year* of the nomination but not the exact day, so an author may have been a year younger than our calculation says. For a question about careers, that's close enough.

## 10. Each author's first nomination

Many authors were nominated more than once, but we want the age at their **first** nomination. We can find it in two steps:

1. **Sort** the books by year with `.sort_values("Year")`, so every author's earliest book comes before their later ones.
2. **Keep the first row for each author** with `.drop_duplicates("Author")`. When an author appears more than once, it keeps the first appearance and drops the rest. After sorting, that first appearance is their earliest nomination.

In [ ]:
books_by_year = known_books.sort_values("Year")
first_nominations = books_by_year.drop_duplicates("Author")

print(len(first_nominations), "authors")
print(len(authors) - len(first_nominations), "authors left out because their date of birth is unknown")

Let's check that it worked for an author with several nominations. Emma Donoghue was nominated in 2008, 2016, and 2020:

In [ ]:
first_nominations[first_nominations["Author"] == "Emma Donoghue"]

Who were the youngest and oldest first-time nominees? Sorting by `Age` puts the youngest first. `.head()` shows the first rows (the youngest), and `.tail()` shows the last rows (the oldest).

In [ ]:
by_age = first_nominations.sort_values("Age")

by_age[["Author", "Year", "Age"]].head()

In [ ]:
by_age[["Author", "Year", "Age"]].tail()

In Week 6 we learned that the **median**, the value in the middle when everything is lined up in order, describes a "typical" value well, especially when a few unusual values could pull the average up or down. pandas calculates both for us.

In [ ]:
median_age = first_nominations["Age"].median()
mean_age = first_nominations["Age"].mean()

print("Median age at first nomination:", median_age)
print("Average age at first nomination:", round(mean_age, 1))

## 11. A chart of ages: the histogram

For a column of numbers like ages, the natural chart is a **histogram**. It divides the ages into ranges, called **bins** (here, 5 years wide: 25–29, 30–34, and so on), and draws one bar for each bin showing how many authors fall into it.

`bins=range(25, 90, 5)` sets where the bins start: 25, 30, 35, and so on. `ax.axvline()` draws a vertical line, which we use to mark the median.

In [ ]:
ax = first_nominations["Age"].plot.hist(bins=range(25, 90, 5), color="#2a78d6", edgecolor="white")

ax.axvline(median_age, color="black", linewidth=1)   # mark the median

ax.set_title("The typical author is first nominated for the Giller in their 40s")
ax.set_xlabel("Age at first Giller nomination")
ax.set_ylabel("Number of authors")
plt.show()

**Discussion:**
- Is the Giller a prize for established writers or for new voices? How does this chart help you argue either side?
- We left out 50 authors whose date of birth is unknown. Which authors are most likely to be missing a date of birth from Wikipedia and other sources? Could leaving them out make the typical age look older or younger than it really is?
- The oldest first-time nominee, Anne Hébert, was nominated in 1999 for an English *translation* of a novel she first published in French. How might translation affect the age at which an author is nominated for the Giller?

## 12. How old are authors when they win?

Because we calculated an age for every book, comparing winners takes only a filter. The `Status` column marks winning books as `winner` or `winner (tie)`, and we'll include both. Each winning book is one row, so an author who wins more than once appears more than once.

In [ ]:
winners = known_books[known_books["Status"].isin(["winner", "winner (tie)"])]

print(len(winners), "winning books")
print("Median age when winning:", winners["Age"].median())
print("Average age when winning:", round(winners["Age"].mean(), 1))

**Discussion:** Compare the median age at first nomination with the median age when winning. Are winners older, younger, or about the same age as first-time nominees? Remember that these are different groups of books: the first group includes every author with a known date of birth, but only some books win, and winning an award is not the same as being nominated.

## 13. Try it yourself

### Birthplaces and residences

1. Use `.value_counts()` to find out how many nominees live outside Canada.
2. Make a filtered DataFrame of the nominees who **live** outside Canada, and draw a chart of where they live. Give it a title that states what you found.
3. The `Nominations` column holds numbers, so we can compare it with `>=` instead of `==`. Make a filtered DataFrame of the authors with 3 or more nominations, and print their names and nominations. (Hint: `filtered[["Author", "Nominations"]]` picks two columns at once.)
4. **Challenge:** Who was born in Canada but lives elsewhere? You'll need two conditions at once. In pandas, `&` plays the role of `and`, and each condition needs its own parentheses:
   ```python
   authors[(condition 1) & (condition 2)]
   ```

### Ages

5. Draw a histogram of the ages of the winners. How does it compare with the histogram in section 11?
6. Change the bins in a histogram to be 10 years wide. Does the chart tell the same story? Which bin width do you prefer, and why?
7. Use filtering to compare the median age of first-time nominees **before 2006** with the median age **from 2006 on**. Are first-time nominees getting younger, older, or staying about the same?
8. Make a list of the authors who were first nominated before the age of 35. Do you recognize any of their names?
9. **Challenge:** A **scatter plot** draws one dot per row, using two numeric columns. Try `first_nominations.plot.scatter(x="Year", y="Age")`. What pattern, if any, do you see? Add a title and axis labels.

In [ ]:
# Write your code here